In [81]:
import sys

origin = 'data/Aldhani/eoagritwin/'
sys.path.append('/home/potzschf/repos/')

from helperToolz.polygons_to_labels import *
from helperToolz.helpsters import *
from helperToolz.dicts_and_lists import * 

gtiff_driver = gdal.GetDriverByName('GTiff')
colorList = ['BLU', 'GRN', 'RED', 'BNR']
chip_size = 256
samples_per_tile = 12

In [82]:
for state, year in zip(['BYN','BW'], [2020, 2021]):
    outFolder = path_safe(f'/data/Aldhani/eoagritwin/fields/output/trash/for_Shreya/{state}/{year}/')
    tiles = get_forcetiles_range(getFilelist(f'/data/Aldhani/eoagritwin/force/output/{state}/{year}/', '.tif', deep=True))
    for tile in tiles:
        force_tifs = getFilelist(f'/data/Aldhani/eoagritwin/force/output/{state}/{year}/{tile}/FBM/', '.tif')

        ds= gdal.Open(force_tifs[0])
        XX = ds.RasterXSize
        YY = ds.RasterYSize
        gt = ds.GetGeoTransform()
        prj = ds.GetProjection()

        for month in [3,6,8]:# range(3,9,1): # nur march, june, august
            monthly = [tif for tif in force_tifs if f'-{month:02d}.tif' in tif]
            MM = (INT_TO_MONTH[f'{month:02d}'])
            # export labels as tif chips
            out_ds = gtiff_driver.Create((f'{outFolder}{state}_{year}_{MM}.tif'), XX, YY, 4, gdal.GDT_Float32)

            out_ds.SetGeoTransform(gt)
            out_ds.SetProjection(prj)


            for band, color in enumerate(colorList):
                for file in monthly:
                    if color == file.split('_SEN2H_')[-1].split('_FBM')[0]:
                        dum_ds = gdal.Open(file)
                        arr = dum_ds.GetRasterBand(1).ReadAsArray()
                        out_ds.GetRasterBand(band + 1).WriteArray(arr)
                        out_ds.GetRasterBand(band + 1).SetDescription(color)

            del out_ds

        # cut them in chipsize * chipsize chips
        tfiles = [tfile for tfile in getFilelist(outFolder, '.tif') if all(substr in tfile for substr in [str(year), state])]
        
        ds = gdal.Open(tfiles[0])
        arr = stackReader(tfiles[0])
        row_col_ind =get_row_col_indices(chip_size, 0, arr.shape[0], arr.shape[1])
        row_start = row_col_ind[0]
        row_end   = row_col_ind[1]
        col_start = row_col_ind[2]
        col_end   = row_col_ind[3]

        geoTF = ds.GetGeoTransform()
        prj = ds.GetProjection()

        mm = tfile.split('_')[-1].split('.')[0]
        chip_path = path_safe(f'{outFolder}{mm}/')

        # draw the random row - col combinations --> therefore, we only write away the samples not all chips for tile

        total_combs = len(row_end) * len(col_end)
        # sample indices in the flattened grid
        idx = np.random.choice(total_combs, size=samples_per_tile, replace=False)

        for tfile in tfiles:
            ds = gdal.Open(tfile)
            arr = stackReader(tfile)
            row_col_ind =get_row_col_indices(chip_size, 0, arr.shape[0], arr.shape[1])
            row_start = row_col_ind[0]
            row_end   = row_col_ind[1]
            col_start = row_col_ind[2]
            col_end   = row_col_ind[3]

            geoTF = ds.GetGeoTransform()
            prj = ds.GetProjection()

            mm = tfile.split('_')[-1].split('.')[0]
            chip_path = path_safe(f'{outFolder}{mm}/')

            counter = 0
            for i in range(len(row_end)):
                for j in range(len(col_end)):
                    if counter in idx:
                        # export as tif chips
                        out_ds = gtiff_driver.Create(f'{chip_path}{state}_{year}_{mm}_{tile}_rs{row_start[i]:04d}_cs{col_start[j]:04d}.tif',
                                                    int(chip_size), int(chip_size), arr.shape[-1], gdal.GDT_Float32)
                        # change the Geotransform for each chip
                        geotf = list(geoTF)
                        # get column and rows from filenames
                        geotf[0] = geotf[0] + geotf[1] * col_start[j]
                        geotf[3] = geotf[3] + geotf[5] * row_start[i]
                        
                        out_ds.SetGeoTransform(tuple(geotf))
                        out_ds.SetProjection(prj)
                        for b in range(arr.shape[-1]):
                            out_ds.GetRasterBand(b+1).WriteArray(arr[row_start[i]:row_end[i], col_start[j]:col_end[j], b])
                            out_ds.GetRasterBand(b+1).SetDescription(colorList[b]) 
                        del out_ds
                    
                        counter += 1
                    else:
                        counter += 1
                        continue
                    

In [83]:
### rename samples
all_chips_March = []
all_chips_June = []
all_chips_August = []

for state, year in zip(['BYN', 'BW'], [2020, 2021]):
    all_chips_March.append(getFilelist(f'/data/Aldhani/eoagritwin/fields/output/trash/for_Shreya/{state}/{year}/March/', '.tif', deep=True))
    all_chips_June.append(getFilelist(f'/data/Aldhani/eoagritwin/fields/output/trash/for_Shreya/{state}/{year}/June/', '.tif', deep=True))
    all_chips_August.append(getFilelist(f'/data/Aldhani/eoagritwin/fields/output/trash/for_Shreya/{state}/{year}/August/', '.tif', deep=True))

all_chips_March = [chips for all in all_chips_March for chips in all]
all_chips_June = [chips for all in all_chips_June for chips in all]
all_chips_August = [chips for all in all_chips_August for chips in all]

all_chips_March.sort()
all_chips_June.sort()
all_chips_August.sort()


In [85]:
# mix the chips and rename
outPath = f'/data/Aldhani/eoagritwin/fields/output/trash/for_Shreya/Sample/'
mixer = np.random.choice(len(all_chips_March), size=len(all_chips_March), replace=False)
print(len(mixer))
for randcount, idx in enumerate(mixer):
    if all_chips_March[idx].split('March')[-1] == all_chips_June[idx].split('June')[-1] and all_chips_March[idx].split('March')[-1] == all_chips_August[idx].split('August')[-1]:
        shutil.copy(all_chips_March[idx], path_safe(f'{outPath}Chip_{randcount+1}_March.tif'))
        shutil.copy(all_chips_June[idx], path_safe(f'{outPath}Chip_{randcount+1}_June.tif'))
        shutil.copy(all_chips_August[idx], path_safe(f'{outPath}Chip_{randcount+1}_August.tif'))
    else:
        raise ValueError('sth wrong')

2040
